<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/dentistry/lecture_5/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%96_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 5: Современные NLP-модели в клинической медицине (стоматология и смежные области)

## Введение

В лекции №5 мы познакомились с тем, как современные NLP-модели (BERT, GPT и их открытые аналоги 2025–2026 годов) могут применяться в клинической медицине и, в частности, в стоматологии. Мы разобрали:

- как слова превращаются в векторы (эмбеддинги) и что это даёт для понимания смысла медицинских текстов,
- как работает архитектура трансформера и механизм внимания,
- в чём различие между BERT (понимание) и GPT (генерация),
- какие открытые модели доступны для работы с русскоязычными медицинскими текстами,
- как извлекать симптомы, анализировать эмоции (в том числе страх и боль), искать похожие клинические случаи,
- какие этические риски сопровождают использование LLM в здравоохранении.

Теперь вам предстоит применить эти знания на практике.

**Цель работы** — закрепить навыки работы с предобученными моделями:
- загрузка моделей с Hugging Face,
- анализ эмоций в тексте пациента (страх, тревога, боль),
- поиск семантически близких клинических случаев,
- извлечение симптомов из жалоб,
- оценка неотложности состояния,
- критическая оценка этических аспектов использования LLM в медицине.

---

## Подготовка рабочей среды

Перед началом работы установите необходимые библиотеки (в терминале или командной строке):

```python
!pip install transformers torch sentence-transformers scikit-learn pandas numpy
```

Для работы с моделями может потребоваться **около 2–3 ГБ свободного места** для загрузки моделей. Если у вас ограниченный интернет-трафик или медленное соединение, вы можете работать с примерами кода в теории, не запуская их локально, или использовать более лёгкие модели (например, `rubert-tiny` вместо `rubert-large`).

Импортируйте необходимые модули:

```python
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
```

## Часть 1. Теоретические вопросы (для самопроверки)

Перед выполнением практических заданий письменно ответьте на следующие вопросы. Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое векторное представление слов (word embedding)? Почему оно важно для понимания медицинских текстов моделями? Приведите пример из стоматологии.

2. В чём основное отличие трансформера от предыдущих архитектур (RNN, LSTM)? Почему это важно для анализа жалоб пациентов?

3. Что такое механизм внимания (attention)? Приведите аналогию из врачебной практики (например, как стоматолог выделяет ключевые симптомы).

4. В чём разница между само-вниманием (self-attention) и обычным вниманием?

5. Что такое предобучение (pre-training) и дообучение (fine-tuning)? Проведите аналогию с медицинским образованием (базовое обучение и ординатура).

6. В чём ключевое различие между BERT и GPT? Для каких задач в стоматологии подходит каждая модель (например, классификация жалоб и генерация рекомендаций)?

7. Назовите три открытые модели 2025–2026 годов, которые можно применить для анализа русскоязычных медицинских текстов. Какие задачи они решают?

8. Что такое RAG (Retrieval-Augmented Generation) и как он помогает бороться с галлюцинациями в клиническом контексте?

9. Какие этические риски возникают при использовании NLP-моделей в медицине? Как их минимизировать?

10. **Рефлексивный вопрос:** как вы видите баланс между автоматизацией (модели ИИ) и человеческим контролем в стоматологии? Что должно оставаться за врачом?

---

## Часть 2. Практические задания на Python

Все задания выполняйте в Jupyter Notebook или отдельном Python-скрипте. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов с точки зрения врача.

---

### Задание 1. Семантический анализ: поиск соседей для клинических терминов

**Описание.** В этом задании вы познакомитесь с векторными эмбеддингами на практике. Вы загрузите предобученную модель Word2Vec (или используете Sentence Transformers) и найдёте слова, семантически близкие к стоматологическим терминам.

**Требуется:**

1. Загрузите русскоязычную модель Word2Vec (`word2vec-ruscorpora-300`) через библиотеку `gensim.downloader`.

2. Найдите 10 слов, наиболее близких по смыслу к слову **«пульпит»**. Выведите их с коэффициентами сходства.

3. Найдите 10 слов, наиболее близких к слову **«периодонтит»**. Сравните полученные списки. Есть ли пересечения? Что это говорит о семантической близости этих заболеваний?

4. Проверьте семантическое расстояние между парами слов:
   - «пульпит» и «периодонтит»
   - «пульпит» и «отбеливание»
   - «периодонтит» и «абсцесс»
   - «периодонтит» и «отбеливание»

5. Найдите «лишнее» слово в списке: `["кариес", "пульпит", "периодонтит", "отбеливание", "гингивит"]`. Какое слово выбивается и почему?

**Интерпретация для врача:** что означает семантическая близость слов в медицинском тексте? Как это может помочь в автоматической категоризации жалоб и поиске похожих случаев?

```python
# Ваш код решения задачи:
```

---

### Задание 2. Анализ эмоций в тексте пациента (страх, тревога, боль)

**Описание.** Используйте модель `ilyali034/rubert-emotion-ru-large` для анализа эмоционального содержания текста пациента. Модель классифицирует текст по 10 базовым эмоциям Изарда (радость, печаль, гнев, энтузиазм, удивление, отвращение, страх, вина, стыд, нейтральное). Это поможет оценить уровень дентофобии и эмоционального напряжения перед лечением.

**Текст для анализа:**

> *«Я очень боюсь идти к стоматологу. У меня паника, сердце колотится, руки трясутся. Я откладывал этот визит уже полгода, но зуб болит так, что спать не могу. Мне стыдно, что я такой трус, но ничего не могу с собой поделать.»*

**Требуется:**

1. Загрузите модель с Hugging Face с помощью `pipeline("text-classification", model="ilyali034/rubert-emotion-ru-large")`.

2. Проанализируйте текст пациента. Выведите все эмоции с вероятностями (только те, где вероятность > 0.1).

3. Напишите интерпретацию результатов:
   - Какие эмоции доминируют?
   - Что это говорит о психологическом состоянии пациента?
   - Какие эмоции могут указывать на дентофобию?
   - Какие меры вы бы предприняли как врач (премедикация, седация, консультация психолога)?

4. *Дополнительно:* проанализируйте второй текст (например, пациент спокоен и описывает плановый осмотр) и сравните эмоциональные профили.

```python
# Ваш код решения задачи:
```

---

### Задание 3. Поиск похожих клинических случаев

**Описание.** Используйте модель `sergeyzh/rubert-large-uncased-sts` (Sentence Transformer) для поиска семантически похожих клинических случаев. Это может помочь врачу находить в архиве случаи с похожей симптоматикой и сравнивать тактики лечения.

**Дано.** База из 10 анонимизированных клинических случаев (краткие описания симптомов, характерные для стоматологии и смежных областей):

```python
clinical_cases = [
    "Пациент жалуется на острую боль в зубе при накусывании, отёк десны",
    "Больной отмечает ноющую боль, усиливающуюся ночью, чувствительность к горячему",
    "Пациент сообщает о кровоточивости дёсен при чистке зубов, неприятном запахе изо рта",
    "Пациент чувствует себя хорошо, зуб не беспокоит, плановый осмотр",
    "Жалобы на подвижность зуба, гноетечение из десны, температура 37.5",
    "Пациент говорит о повышенной чувствительности к холодному, боль стихает после устранения раздражителя",
    "Больной сообщает о припухлости щеки, затруднении открывания рта, общем недомогании",
    "Пациент отмечает улучшение после лечения, боль уменьшилась, отёк спал",
    "Жалобы на сухость во рту, жжение языка, нарушение вкуса",
    "Пациент описывает боль в челюсти, щёлканье при открывании рта, головные боли"
]
```

**Новый пациент:**
```python
new_patient = "Пациент жалуется на боль в зубе при жевании, отёк десны и припухлость щеки"
```

**Требуется:**

1. Загрузите модель `sergeyzh/rubert-large-uncased-sts`.

2. Получите эмбеддинги для всех клинических случаев и для нового пациента.

3. Вычислите косинусное сходство между новым пациентом и каждым случаем из базы.

4. Отсортируйте случаи по убыванию сходства. Выведите топ-3 наиболее похожих случая с коэффициентами сходства.

5. Напишите интерпретацию: что общего у найденных случаев? Какие диагнозы можно предположить? Какую информацию это может дать практикующему врачу?

```python
# Ваш код решения задачи:
```

---

### Задание 4. Извлечение симптомов и оценка неотложности состояния

**Описание.** В этом задании вы используете модель `astromis/presuisidal_rubert` (если она доступна) для оценки наличия тревожных маркеров в тексте (например, суицидальные мысли при хронической боли), а также вручную выделите симптомы (имитация работы модели) и оцените срочность обращения.

**Текст сообщения пациента в онлайн-чат клиники:**

> *«Здравствуйте, у меня зуб болит уже третий день, особенно когда кусаю. Десна опухла, щека припухла, температура 37.4. Боль ноющая, ночью не могу спать. Обезболивающее помогает ненадолго. Что мне делать?»*

**Требуется:**

1. Загрузите модель `astromis/presuisidal_rubert` (или аналогичную) и проверьте, есть ли в тексте признаки суицидального риска (в данном тексте их, скорее всего, нет, но вы должны убедиться). Выведите результат.

2. **Вручную (имитируя работу модели):** выпишите все симптомы, упомянутые в тексте, и разделите их на категории:
   - **Болевые** (характер боли: острая, ноющая, ночная, при накусывании)
   - **Воспалительные** (отёк десны, припухлость щеки)
   - **Системные** (температура)
   - **Эмоциональные** (тревога, страх – если есть)

3. **Оцените срочность** обращения: является ли состояние неотложным? Какие диагнозы можно предположить (острый периодонтит, периостит, абсцесс)? Какие рекомендации вы дадите пациенту до визита?

4. Напишите краткое резюме (1 абзац) с обоснованием вашего решения.

```python
# Ваш код решения задачи:
```

---

### Задание 5. Генерация резюме клинического случая (GPT-подход)

**Описание.** В этом задании вы познакомитесь с генеративной способностью LLM. Поскольку запуск больших моделей требует значительных ресурсов, мы будем использовать простой подход: вы вручную составите резюме клинического случая, а затем сравните его с тем, как могла бы сделать модель (имитация).

**Задача.** У вас есть запись из амбулаторной карты (фрагмент). Составьте **краткое резюме** (10–15 предложений), выделяя:
- основные жалобы пациента,
- данные объективного осмотра,
- предполагаемый диагноз,
- план лечения,
- рекомендации.

**Запись (анонимизированная):**

> «Пациент К., 35 лет, обратился с жалобами на боли в 16 зубе, усиливающиеся при накусывании, отёк десны в области 16, припухлость щеки справа, повышение температуры до 37.5. Со слов пациента, боль появилась 3 дня назад, постепенно усиливалась, приём анальгетиков даёт кратковременный эффект. Объективно: лицо асимметрично за счёт отёка мягких тканей правой щеки, открывание рта ограничено (2 пальца), слизистая в проекции 16 зуба гиперемирована, отёчна, пальпация резко болезненна, перкуссия 16 резко положительна. Зуб 16 с глубокой кариозной полостью, зондирование дна безболезненно (пульпа некротизирована), подвижность I степени. На рентгенограмме: у верхушки дистального корня 16 определяется очаг разрежения костной ткани с нечёткими контурами размером 5×4 мм. Диагноз: хронический верхушечный периодонтит 16 в стадии обострения, периостит нижней челюсти справа. План: раскрытие полости 16, эндодонтическое лечение, назначение антибиотиков (амоксициллин/клавуланат), НПВС, холод местно. Рекомендовано: при неэффективности — хирургическое лечение (удаление 16).»

**Требования к резюме:**

- Используйте клиническую лексику (кариозная полость, некроз пульпы, перкуссия, рентгенограмма, периостит и т.д.).
- Укажите данные осмотра и диагностики.
- Сформулируйте план лечения и рекомендации.

Напишите рефлексию (1 абзац): что было легко/сложно при составлении резюме? Где могла бы ошибиться GPT-модель?

```python
# Ваше резюме и рефлексия (текст, не код):
```

---

### Задание 6. Сравнение моделей: BERT vs GPT для клинических задач

**Описание.** В этом теоретико-практическом задании вы сравните два подхода — BERT (понимание) и GPT (генерация) — для двух клинических задач.

**Задача А. Классификация (BERT-подход)**

Даны 5 текстов обращений пациентов. Классифицируйте их как «Острое состояние (неотложная помощь)» или «Плановое обращение».

Тексты для классификации:
1. «Хочу записаться на профилактический осмотр, ничего не беспокоит.»
2. «У меня зуб болит при накусывании, десна опухла, температура 37.8.»
3. «После лечения всё хорошо, зуб не болит, но хочу уточнить по уходу.»
4. «Сильная боль в зубе, отёк щеки, трудно открывать рот.»
5. «Заметил кариес на переднем зубе, но боли нет.»

Проведите классификацию **вручную** (имитация работы BERT). Для каждого текста укажите, почему вы отнесли его к той или иной категории (какие слова-маркеры использовали?).

**Задача Б. Генерация (GPT-подход)**

Для текста 4 («Сильная боль в зубе, отёк щеки, трудно открывать рот») напишите, как бы мог ответить GPT-бот клиники (имитация). Какие фразы он мог бы использовать? Какие вопросы задать (аллергии, принимаемые препараты)? Какие рекомендации дать (срочно обратиться, холод местно, обезболивающее)? Какие предостережения нужны (не греть, не принимать пищу на эту сторону)?

**Задача В. Сравнение**

Напишите эссе (1 страница) на тему: *«Когда в стоматологии лучше использовать BERT, а когда — GPT?»* Опишите преимущества и ограничения каждого подхода. Приведите два конкретных примера из практики (можно гипотетических) для каждого подхода.

```python
# Ваш ответ (текст, не код):
```

---

## Часть 3. Этический анализ

**Задание 7. Этический протокол использования LLM в клинике**

**Описание.** Представьте, что вы — главный врач стоматологической клиники. Вам предлагают внедрить систему на основе открытой LLM (например, `KhazarAI/MentalChat-16K` или `Horiznsky/Serenity-Llama-3.2-3B-Counsel`, адаптированные для медицинских консультаций) для:
1. Автоматического анализа сообщений пациентов в онлайн-чате (определение эмоций, симптомов, срочности).
2. Генерации предварительных рекомендаций и маршрутизации пациентов (к терапевту, хирургу, ортодонту).

**Напишите этический протокол** (объёмом 2–3 страницы), в котором осветите:

1. **Информированное согласие** — что именно пациент должен знать об использовании ИИ? Как вы объясните ему ограничения модели (галлюцинации, возможность ошибки)?

2. **Конфиденциальность и безопасность** — как будут храниться данные? Будет ли использоваться облако или on-premise развёртывание? Кто имеет доступ к данным и результатам анализа? Особое внимание уделите хранению персональных данных и медицинской тайне (ФЗ-152, врачебная тайна).

3. **Человеческий контроль** — как организовать верификацию результатов модели? Кто отвечает за принятие финальных решений? Как избежать автоматического смещения (слепого доверия машине)?

4. **Обработка ошибок и неотложных состояний** — что делать, если модель пропустит признаки острого воспаления (периостит, абсцесс)? Что делать, если модель ложно направит пациента к узкому специалисту?

5. **Прозрачность и интерпретируемость** — как сделать работу модели понятной для пациента? Как вы будете объяснять, почему модель выдала тот или иной результат?

6. **Обучение персонала** — кто и как будет обучаться работе с системой?

```python
# Ваш ответ (текст, не код):
```

---

## Часть 4. Комплексное задание (повышенной сложности) — по желанию

**Задание 8. Fine-Tuning небольшой модели для классификации жалоб**

**Описание.** Если вы знакомы с основами машинного обучения и имеете доступ к GPU (или используете Google Colab), попробуйте выполнить дообучение (fine-tuning) модели `rubert-tiny` на небольшом датасете стоматологических жалоб.

**Инструкция (упрощённая):**

1. Создайте синтетический датасет из 30–50 коротких фраз (по 10 фраз на 3–4 категории: острая боль/воспаление, плановый осмотр, чувствительность, после лечения).

2. Загрузите модель `cointegrated/rubert-tiny`.

3. С помощью `Trainer` из библиотеки `transformers` выполните дообучение на 2–3 эпохи.

4. Оцените качество на тестовых примерах.

5. Напишите рефлексию (1 страница): что было сложно? Какие ограничения вы заметили? Какой объём данных нужен для дообучения в реальной клинической задаче?

Это задание требует предварительного знакомства с PyTorch и Hugging Face Trainer. Если вы не знакомы — пропустите.

```python
# Ваш код решения задачи:
```

## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (семантические соседи) | 10% | Корректность кода, интерпретация |
| Задание 2 (эмоциональный анализ) | 15% | Корректность кода, глубина интерпретации |
| Задание 3 (поиск похожих случаев) | 10% | Корректность кода, анализ результатов |
| Задание 4 (извлечение симптомов, неотложность) | 15% | Полнота выделения симптомов, оценка срочности |
| Задание 5 (генерация резюме) | 10% | Качество резюме, рефлексия |
| Задание 6 (сравнение BERT и GPT) | 10% | Глубина анализа, аргументированность |
| Задание 7 (этический протокол) | 15% | Полнота, практичность, аргументированность |
| Задание 8 (fine-tuning, бонус) | +5% | Корректность, рефлексия |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb` или Python `.py`) со всеми заданиями, кодом и текстовыми комментариями.
- Эссе и этический протокол (Задания 6 и 7) приложите в виде текстовых ячеек в Notebook или отдельного документа (`.txt` или `.pdf`).
- Убедитесь, что код выполняется без ошибок (укажите версии библиотек при необходимости).
- Все результаты анализа сопровождайте интерпретацией с точки зрения врача.

---

## Заключение

Данная практическая работа проведёт вас через полный цикл работы с современными NLP-моделями — от базовых эмбеддингов до этических размышлений об использовании ИИ в клинической практике. Вы не только освоите инструменты, но и научитесь **критически оценивать** их применение в стоматологии и смежных областях.

**Главный вывод:** современные NLP-модели — это мощные помощники, но они не заменяют клиническое мышление. Ответственность за решения всегда остаётся за врачом.

---

**Срок выполнения: 2 недели.**